In [1]:
from lxml import etree
import pandas as pd
import requests
import time

In [2]:
def fetch_html(url: str) -> str:
    try:
        request_headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.5",
            "Connection": "keep-alive"
        }
        response = requests.get(url, headers=request_headers, timeout=10)
        response.raise_for_status()
        return response.text
    except requests.RequestException as e:
        print(f"An error occurred: {e}")
        return ""

In [3]:
def generate_base_links_endpoint(page_index:int) -> str:
    return 'https://bina.az/baki/alqi-satqi/menziller?page=' + str(page_index)

In [4]:
def extract_data_from_html(html_content: str):
    tree = etree.HTML(html_content)
    
    items_list = tree.xpath('(//div[contains(@class, "items_list")])[2]')
    data = []

    if items_list:
        items_list = items_list[0]
        items = items_list.xpath('.//div[contains(@class, "items-i")]')

        for item in items:
            item_link = item.xpath('.//a[contains(@class, "item_link")]/@href')
            item_link = item_link[0] if item_link else None

            preview = item.xpath('.//div[contains(@class, "preview")]')
            if preview:
                preview = preview[0]
                bill_of_sale = bool(preview.xpath('.//div[contains(@class, "bill_of_sale")]'))
                mortgage = bool(preview.xpath('.//div[contains(@class, "mortgage")]'))
                repair = bool(preview.xpath('.//div[contains(@class, "repair")]'))
                products_label = preview.xpath('.//div[contains(@class, "products-label")]/text()')
                products_label = products_label[0].strip() if products_label else None
            else:
                bill_of_sale = mortgage = repair = False
                products_label = None

            card_params = item.xpath('.//div[contains(@class, "card_params")]')
            if card_params:
                card_params = card_params[0]
                price_spans = card_params.xpath('.//div[contains(@class, "price")]//span/text()')
                price = ''.join([span.strip() for span in price_spans]).strip() if price_spans else None

                location = card_params.xpath('.//div[contains(@class, "location")]/text()')
                location = location[0].strip() if location else None

                ul_name = card_params.xpath('.//ul[contains(@class, "name")]')
                if ul_name:
                    ul_name = ul_name[0]
                    room_count = ul_name.xpath('./li[1]/text()')
                    room_count = room_count[0].strip() if room_count else None

                    area = ul_name.xpath('./li[2]/text()')
                    area = area[0].strip() if area else None

                    floor = ul_name.xpath('./li[3]/text()')
                    floor = floor[0].strip() if floor else None
                else:
                    room_count = area = floor = None

                card_footer = card_params.xpath('.//div[contains(@class, "city_when")]/text()')
                card_footer = [cf.strip() for cf in card_footer if cf.strip()] if card_footer else []
                city = card_footer[0] if len(card_footer) > 0 else None
            else:
                price = location = room_count = area = floor = city = uploaded_time = None

            data.append({
                "post_href": item_link,
                "bill_of_sale_available": bill_of_sale,
                "mortgage_available": mortgage,
                "repair_available": repair,
                "products_label": products_label,
                "price": price,
                "location": location,
                "room_count": room_count,
                "area": area,
                "floor": floor,
                "city_and_added_time": city,
            })
    
    return data


In [5]:
def process_pages(start: int, end: int):
    all_data = []

    for page_index in range(start, end + 1):
        try:
            url = generate_base_links_endpoint(page_index)
            html_content = fetch_html(url)
            
            if html_content:
                data = extract_data_from_html(html_content)
                all_data.extend(data)
                print(f"Page {page_index} processed.")
            
            time.sleep(10)
        except Exception as e:
            print(f"Error processing page {page_index}: {e}")
    
    return all_data

In [6]:
start_page = 1
end_page = 20

all_data = process_pages(start_page, end_page)

df = pd.DataFrame(all_data)
csv_file = 'data/scraped_data_team_43.csv'
df.to_csv(csv_file, index=False, encoding='utf-8')
print(f"Data successfully written to {csv_file}")

Page 1 processed.
Page 2 processed.
Page 3 processed.
Page 4 processed.
Page 5 processed.
Page 6 processed.
Page 7 processed.
Page 8 processed.
Page 9 processed.
Page 10 processed.
Page 11 processed.
Page 12 processed.
Page 13 processed.
Page 14 processed.
Page 15 processed.
Page 16 processed.
Page 17 processed.
Page 18 processed.
Page 19 processed.
Page 20 processed.
Data successfully written to data/scraped_data_team_43.csv
